# Table 2 · Agreement with human ratings

BVCC test system-level SRCC and utterance MSE, and mean system-level SRCC over 12 out-of-domain test sets, for every
predictor (mean ± sd over 3 seeds). Recomputes E2: every predictor scores every test set (30 jobs, ~3 min each).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e2']]
print(len(KEYS), "runs:", ", ".join(KEYS))

30 runs: e2_R0_s0, e2_R0_s1, e2_R0_s2, e2_R1_s0, e2_R2_s0, e2_R3_s0, e2_baseline_noise_s0, e2_baseline_pgd_s0, e2_baseline_random_s0, e2_R1_s1, e2_R2_s1, e2_R3_s1, e2_baseline_noise_s1, e2_baseline_pgd_s1, e2_baseline_random_s1, e2_R1_s2, e2_R2_s2, e2_R3_s2, e2_baseline_noise_s2, e2_baseline_pgd_s2, e2_baseline_random_s2, e2_abl_tau0p1_s0_s0, e2_abl_margin0p1_s0_s0, e2_abl_headonly_s0_s0, e2_abl_margin1p0_s0_s0, e2_abl_prefinput_s0_s0, e2_abl_tau0p5_s0_s0, e2_R3ms_s0, e2_R3ms_s1, e2_R3ms_s2


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    30
Name: count, dtype: int64

In [6]:
print((PAPER / "tables" / "table2.tex").read_text())

% generated by analysis/paper_assets.py
\begin{tabular}{l c c c}
\toprule
& \multicolumn{2}{c}{BVCC test} & out-of-domain \\
\cmidrule(lr){2-3}
Predictor & sys SRCC & utt MSE & mean sys SRCC \\
\midrule
$R_0$ (no defense) & 0.922$\pm$0.004 & 0.234$\pm$0.011 & 0.731$\pm$0.006 \\
$R_1$ & 0.911$\pm$0.009 & 0.272$\pm$0.034 & 0.690$\pm$0.014 \\
$R_2$ & 0.918$\pm$0.003 & 0.272$\pm$0.006 & 0.682$\pm$0.008 \\
$R_3$ (EnCodec loop) & 0.920$\pm$0.001 & 0.272$\pm$0.022 & 0.637$\pm$0.046 \\
$R_3^{\mathrm{multi}}$ (multi-space) & 0.914$\pm$0.003 & 0.277$\pm$0.004 & 0.552$\pm$0.015 \\
noise pairs & 0.917$\pm$0.004 & 0.275$\pm$0.021 & 0.722$\pm$0.013 \\
PGD pairs & 0.909$\pm$0.001 & 0.274$\pm$0.023 & 0.635$\pm$0.009 \\
random-latent pairs & 0.922$\pm$0.004 & 0.298$\pm$0.049 & 0.696$\pm$0.010 \\
\bottomrule
\end{tabular}



In [7]:
print(repro.diff_text(PAPER / "tables" / "table2.tex", COMMITTED_PAPER / "tables" / "table2.tex"))

identical to the committed file


In [8]:
display(Markdown((OUT / "table2_e2.md").read_text()))

| corpus       | split         | predictor        | utt_mse             | utt_lcc             | utt_srcc            | sys_mse             | sys_lcc             | sys_srcc             |
|:-------------|:--------------|:-----------------|:--------------------|:--------------------|:--------------------|:--------------------|:--------------------|:---------------------|
| bvcc         | test          | R0               | 0.234 ± 0.011 (n=3) | 0.880 ± 0.005 (n=3) | 0.878 ± 0.005 (n=3) | 0.113 ± 0.010 (n=3) | 0.926 ± 0.004 (n=3) | 0.922 ± 0.004 (n=3)  |
| bvcc         | test          | R1               | 0.272 ± 0.034 (n=3) | 0.872 ± 0.006 (n=3) | 0.870 ± 0.007 (n=3) | 0.145 ± 0.025 (n=3) | 0.914 ± 0.009 (n=3) | 0.911 ± 0.009 (n=3)  |
| bvcc         | test          | R2               | 0.272 ± 0.006 (n=3) | 0.876 ± 0.004 (n=3) | 0.874 ± 0.004 (n=3) | 0.142 ± 0.005 (n=3) | 0.922 ± 0.004 (n=3) | 0.918 ± 0.003 (n=3)  |
| bvcc         | test          | R3               | 0.272 ± 0.022 (n=3) | 0.876 ± 0.004 (n=3) | 0.874 ± 0.003 (n=3) | 0.140 ± 0.011 (n=3) | 0.924 ± 0.001 (n=3) | 0.920 ± 0.001 (n=3)  |
| bvcc         | test          | R3ms             | 0.277 ± 0.004 (n=3) | 0.869 ± 0.002 (n=3) | 0.867 ± 0.003 (n=3) | 0.144 ± 0.003 (n=3) | 0.918 ± 0.003 (n=3) | 0.914 ± 0.003 (n=3)  |
| bvcc         | test          | abl_headonly_s0  | 0.333 (n=1)         | 0.857 (n=1)         | 0.860 (n=1)         | 0.185 (n=1)         | 0.912 (n=1)         | 0.910 (n=1)          |
| bvcc         | test          | abl_margin0p1_s0 | 0.299 (n=1)         | 0.872 (n=1)         | 0.870 (n=1)         | 0.168 (n=1)         | 0.917 (n=1)         | 0.912 (n=1)          |
| bvcc         | test          | abl_margin1p0_s0 | 0.251 (n=1)         | 0.875 (n=1)         | 0.872 (n=1)         | 0.124 (n=1)         | 0.923 (n=1)         | 0.920 (n=1)          |
| bvcc         | test          | abl_prefinput_s0 | 0.309 (n=1)         | 0.875 (n=1)         | 0.871 (n=1)         | 0.180 (n=1)         | 0.922 (n=1)         | 0.916 (n=1)          |
| bvcc         | test          | abl_tau0p1_s0    | 0.281 (n=1)         | 0.881 (n=1)         | 0.879 (n=1)         | 0.160 (n=1)         | 0.922 (n=1)         | 0.917 (n=1)          |
| bvcc         | test          | abl_tau0p5_s0    | 0.273 (n=1)         | 0.876 (n=1)         | 0.874 (n=1)         | 0.153 (n=1)         | 0.922 (n=1)         | 0.918 (n=1)          |
| bvcc         | test          | baseline_noise   | 0.275 ± 0.021 (n=3) | 0.872 ± 0.007 (n=3) | 0.869 ± 0.007 (n=3) | 0.143 ± 0.020 (n=3) | 0.921 ± 0.004 (n=3) | 0.917 ± 0.004 (n=3)  |
| bvcc         | test          | baseline_pgd     | 0.274 ± 0.023 (n=3) | 0.871 ± 0.005 (n=3) | 0.868 ± 0.005 (n=3) | 0.147 ± 0.025 (n=3) | 0.916 ± 0.002 (n=3) | 0.909 ± 0.001 (n=3)  |
| bvcc         | test          | baseline_random  | 0.298 ± 0.049 (n=3) | 0.874 ± 0.006 (n=3) | 0.872 ± 0.006 (n=3) | 0.155 ± 0.038 (n=3) | 0.925 ± 0.005 (n=3) | 0.922 ± 0.004 (n=3)  |
| bvcc_ood     | test          | R0               | 2.636 ± 0.592 (n=3) | 0.636 ± 0.032 (n=3) | 0.654 ± 0.023 (n=3) | 2.435 ± 0.584 (n=3) | 0.728 ± 0.027 (n=3) | 0.729 ± 0.013 (n=3)  |
| bvcc_ood     | test          | R1               | 2.862 ± 0.310 (n=3) | 0.646 ± 0.038 (n=3) | 0.631 ± 0.037 (n=3) | 2.610 ± 0.266 (n=3) | 0.750 ± 0.045 (n=3) | 0.707 ± 0.070 (n=3)  |
| bvcc_ood     | test          | R2               | 2.956 ± 0.245 (n=3) | 0.638 ± 0.035 (n=3) | 0.627 ± 0.034 (n=3) | 2.695 ± 0.225 (n=3) | 0.748 ± 0.032 (n=3) | 0.733 ± 0.042 (n=3)  |
| bvcc_ood     | test          | R3               | 3.114 ± 0.161 (n=3) | 0.607 ± 0.025 (n=3) | 0.603 ± 0.033 (n=3) | 2.866 ± 0.177 (n=3) | 0.715 ± 0.022 (n=3) | 0.708 ± 0.014 (n=3)  |
| bvcc_ood     | test          | R3ms             | 3.802 ± 0.145 (n=3) | 0.359 ± 0.022 (n=3) | 0.363 ± 0.023 (n=3) | 3.313 ± 0.088 (n=3) | 0.512 ± 0.024 (n=3) | 0.533 ± 0.030 (n=3)  |
| bvcc_ood     | test          | abl_headonly_s0  | 3.141 (n=1)         | 0.553 (n=1)         | 0.553 (n=1)         | 2.893 (n=1)         | 0.651 (n=1)         | 0.615 (n=1)          |
| bvcc_ood     | test          | abl_margin0p1_s0 | 3.088 (n=1)         | 0.563 (n=1)         | 0.580 (n=1)         | 2.859 (n=1)         | 0.627 (n=1)         | 0.636 (n=1)          |
| bvcc_ood     | test          | abl_margin1p0_s0 | 3.152 (n=1)         | 0.593 (n=1)         | 0.590 (n=1)         | 2.870 (n=1)         | 0.700 (n=1)         | 0.726 (n=1)          |
| bvcc_ood     | test          | abl_prefinput_s0 | 3.375 (n=1)         | 0.584 (n=1)         | 0.626 (n=1)         | 3.088 (n=1)         | 0.663 (n=1)         | 0.759 (n=1)          |
| bvcc_ood     | test          | abl_tau0p1_s0    | 2.870 (n=1)         | 0.628 (n=1)         | 0.621 (n=1)         | 2.624 (n=1)         | 0.721 (n=1)         | 0.690 (n=1)          |
| bvcc_ood     | test          | abl_tau0p5_s0    | 3.038 (n=1)         | 0.582 (n=1)         | 0.610 (n=1)         | 2.798 (n=1)         | 0.674 (n=1)         | 0.678 (n=1)          |
| bvcc_ood     | test          | baseline_noise   | 2.650 ± 0.155 (n=3) | 0.655 ± 0.014 (n=3) | 0.652 ± 0.006 (n=3) | 2.436 ± 0.145 (n=3) | 0.755 ± 0.021 (n=3) | 0.740 ± 0.035 (n=3)  |
| bvcc_ood     | test          | baseline_pgd     | 3.617 ± 0.323 (n=3) | 0.439 ± 0.048 (n=3) | 0.433 ± 0.042 (n=3) | 3.137 ± 0.284 (n=3) | 0.602 ± 0.044 (n=3) | 0.573 ± 0.041 (n=3)  |
| bvcc_ood     | test          | baseline_random  | 2.882 ± 0.239 (n=3) | 0.653 ± 0.022 (n=3) | 0.643 ± 0.027 (n=3) | 2.616 ± 0.212 (n=3) | 0.739 ± 0.038 (n=3) | 0.722 ± 0.060 (n=3)  |
| nisqa        | test_for      | R0               | 0.476 ± 0.117 (n=3) | 0.806 ± 0.004 (n=3) | 0.779 ± 0.010 (n=3) | 0.397 ± 0.113 (n=3) | 0.864 ± 0.011 (n=3) | 0.797 ± 0.015 (n=3)  |
| nisqa        | test_for      | R1               | 0.734 ± 0.108 (n=3) | 0.738 ± 0.029 (n=3) | 0.683 ± 0.032 (n=3) | 0.638 ± 0.107 (n=3) | 0.787 ± 0.031 (n=3) | 0.690 ± 0.032 (n=3)  |
| nisqa        | test_for      | R2               | 0.668 ± 0.036 (n=3) | 0.729 ± 0.018 (n=3) | 0.677 ± 0.022 (n=3) | 0.574 ± 0.037 (n=3) | 0.778 ± 0.025 (n=3) | 0.690 ± 0.021 (n=3)  |
| nisqa        | test_for      | R3               | 0.899 ± 0.200 (n=3) | 0.667 ± 0.060 (n=3) | 0.626 ± 0.046 (n=3) | 0.794 ± 0.190 (n=3) | 0.709 ± 0.071 (n=3) | 0.613 ± 0.066 (n=3)  |
| nisqa        | test_for      | R3ms             | 1.433 ± 0.166 (n=3) | 0.537 ± 0.028 (n=3) | 0.492 ± 0.037 (n=3) | 1.308 ± 0.161 (n=3) | 0.562 ± 0.029 (n=3) | 0.503 ± 0.045 (n=3)  |
| nisqa        | test_for      | abl_headonly_s0  | 0.920 (n=1)         | 0.672 (n=1)         | 0.626 (n=1)         | 0.792 (n=1)         | 0.723 (n=1)         | 0.651 (n=1)          |
| nisqa        | test_for      | abl_margin0p1_s0 | 0.610 (n=1)         | 0.761 (n=1)         | 0.706 (n=1)         | 0.521 (n=1)         | 0.814 (n=1)         | 0.719 (n=1)          |
| nisqa        | test_for      | abl_margin1p0_s0 | 1.070 (n=1)         | 0.586 (n=1)         | 0.609 (n=1)         | 0.944 (n=1)         | 0.618 (n=1)         | 0.573 (n=1)          |
| nisqa        | test_for      | abl_prefinput_s0 | 0.723 (n=1)         | 0.719 (n=1)         | 0.675 (n=1)         | 0.621 (n=1)         | 0.767 (n=1)         | 0.680 (n=1)          |
| nisqa        | test_for      | abl_tau0p1_s0    | 0.707 (n=1)         | 0.714 (n=1)         | 0.672 (n=1)         | 0.612 (n=1)         | 0.758 (n=1)         | 0.677 (n=1)          |
| nisqa        | test_for      | abl_tau0p5_s0    | 0.606 (n=1)         | 0.758 (n=1)         | 0.700 (n=1)         | 0.517 (n=1)         | 0.809 (n=1)         | 0.703 (n=1)          |
| nisqa        | test_for      | baseline_noise   | 0.519 ± 0.033 (n=3) | 0.766 ± 0.008 (n=3) | 0.756 ± 0.015 (n=3) | 0.436 ± 0.027 (n=3) | 0.816 ± 0.007 (n=3) | 0.766 ± 0.021 (n=3)  |
| nisqa        | test_for      | baseline_pgd     | 0.917 ± 0.093 (n=3) | 0.651 ± 0.015 (n=3) | 0.650 ± 0.009 (n=3) | 0.811 ± 0.089 (n=3) | 0.689 ± 0.015 (n=3) | 0.675 ± 0.011 (n=3)  |
| nisqa        | test_for      | baseline_random  | 0.666 ± 0.190 (n=3) | 0.754 ± 0.018 (n=3) | 0.709 ± 0.019 (n=3) | 0.574 ± 0.189 (n=3) | 0.798 ± 0.019 (n=3) | 0.711 ± 0.027 (n=3)  |
| nisqa        | test_livetalk | R0               | 1.264 ± 0.294 (n=3) | 0.776 ± 0.014 (n=3) | 0.775 ± 0.011 (n=3) | 1.130 ± 0.289 (n=3) | 0.821 ± 0.012 (n=3) | 0.798 ± 0.010 (n=3)  |
| nisqa        | test_livetalk | R1               | 1.800 ± 0.160 (n=3) | 0.738 ± 0.028 (n=3) | 0.712 ± 0.037 (n=3) | 1.655 ± 0.154 (n=3) | 0.781 ± 0.033 (n=3) | 0.736 ± 0.033 (n=3)  |
| nisqa        | test_livetalk | R2               | 1.934 ± 0.173 (n=3) | 0.725 ± 0.016 (n=3) | 0.687 ± 0.019 (n=3) | 1.782 ± 0.170 (n=3) | 0.768 ± 0.015 (n=3) | 0.707 ± 0.021 (n=3)  |
| nisqa        | test_livetalk | R3               | 2.266 ± 0.392 (n=3) | 0.662 ± 0.053 (n=3) | 0.627 ± 0.052 (n=3) | 2.078 ± 0.367 (n=3) | 0.714 ± 0.039 (n=3) | 0.647 ± 0.051 (n=3)  |
| nisqa        | test_livetalk | R3ms             | 2.253 ± 0.440 (n=3) | 0.626 ± 0.074 (n=3) | 0.612 ± 0.084 (n=3) | 2.045 ± 0.404 (n=3) | 0.687 ± 0.054 (n=3) | 0.610 ± 0.067 (n=3)  |
| nisqa        | test_livetalk | abl_headonly_s0  | 2.502 (n=1)         | 0.682 (n=1)         | 0.682 (n=1)         | 2.283 (n=1)         | 0.754 (n=1)         | 0.736 (n=1)          |
| nisqa        | test_livetalk | abl_margin0p1_s0 | 1.703 (n=1)         | 0.719 (n=1)         | 0.697 (n=1)         | 1.564 (n=1)         | 0.750 (n=1)         | 0.701 (n=1)          |
| nisqa        | test_livetalk | abl_margin1p0_s0 | 2.754 (n=1)         | 0.599 (n=1)         | 0.586 (n=1)         | 2.482 (n=1)         | 0.681 (n=1)         | 0.633 (n=1)          |
| nisqa        | test_livetalk | abl_prefinput_s0 | 1.939 (n=1)         | 0.674 (n=1)         | 0.630 (n=1)         | 1.765 (n=1)         | 0.722 (n=1)         | 0.648 (n=1)          |
| nisqa        | test_livetalk | abl_tau0p1_s0    | 1.781 (n=1)         | 0.689 (n=1)         | 0.665 (n=1)         | 1.627 (n=1)         | 0.724 (n=1)         | 0.684 (n=1)          |
| nisqa        | test_livetalk | abl_tau0p5_s0    | 1.498 (n=1)         | 0.726 (n=1)         | 0.693 (n=1)         | 1.358 (n=1)         | 0.761 (n=1)         | 0.693 (n=1)          |
| nisqa        | test_livetalk | baseline_noise   | 1.267 ± 0.121 (n=3) | 0.758 ± 0.004 (n=3) | 0.765 ± 0.009 (n=3) | 1.133 ± 0.118 (n=3) | 0.795 ± 0.007 (n=3) | 0.783 ± 0.014 (n=3)  |
| nisqa        | test_livetalk | baseline_pgd     | 1.642 ± 0.139 (n=3) | 0.690 ± 0.013 (n=3) | 0.717 ± 0.008 (n=3) | 1.482 ± 0.136 (n=3) | 0.737 ± 0.007 (n=3) | 0.711 ± 0.014 (n=3)  |
| nisqa        | test_livetalk | baseline_random  | 1.873 ± 0.340 (n=3) | 0.741 ± 0.013 (n=3) | 0.724 ± 0.016 (n=3) | 1.727 ± 0.338 (n=3) | 0.780 ± 0.016 (n=3) | 0.728 ± 0.024 (n=3)  |
| nisqa        | test_p501     | R0               | 0.306 ± 0.018 (n=3) | 0.869 ± 0.005 (n=3) | 0.881 ± 0.007 (n=3) | 0.229 ± 0.029 (n=3) | 0.900 ± 0.008 (n=3) | 0.920 ± 0.013 (n=3)  |
| nisqa        | test_p501     | R1               | 0.377 ± 0.026 (n=3) | 0.851 ± 0.007 (n=3) | 0.877 ± 0.005 (n=3) | 0.292 ± 0.028 (n=3) | 0.879 ± 0.008 (n=3) | 0.920 ± 0.007 (n=3)  |
| nisqa        | test_p501     | R2               | 0.394 ± 0.034 (n=3) | 0.847 ± 0.007 (n=3) | 0.876 ± 0.010 (n=3) | 0.310 ± 0.042 (n=3) | 0.875 ± 0.003 (n=3) | 0.921 ± 0.008 (n=3)  |
| nisqa        | test_p501     | R3               | 0.402 ± 0.053 (n=3) | 0.841 ± 0.005 (n=3) | 0.866 ± 0.003 (n=3) | 0.318 ± 0.058 (n=3) | 0.867 ± 0.010 (n=3) | 0.905 ± 0.007 (n=3)  |
| nisqa        | test_p501     | R3ms             | 0.605 ± 0.092 (n=3) | 0.800 ± 0.012 (n=3) | 0.838 ± 0.016 (n=3) | 0.506 ± 0.101 (n=3) | 0.824 ± 0.015 (n=3) | 0.884 ± 0.016 (n=3)  |
| nisqa        | test_p501     | abl_headonly_s0  | 0.516 (n=1)         | 0.822 (n=1)         | 0.862 (n=1)         | 0.420 (n=1)         | 0.846 (n=1)         | 0.902 (n=1)          |
| nisqa        | test_p501     | abl_margin0p1_s0 | 0.380 (n=1)         | 0.854 (n=1)         | 0.874 (n=1)         | 0.302 (n=1)         | 0.881 (n=1)         | 0.915 (n=1)          |
| nisqa        | test_p501     | abl_margin1p0_s0 | 0.472 (n=1)         | 0.805 (n=1)         | 0.868 (n=1)         | 0.377 (n=1)         | 0.831 (n=1)         | 0.905 (n=1)          |
| nisqa        | test_p501     | abl_prefinput_s0 | 0.395 (n=1)         | 0.853 (n=1)         | 0.883 (n=1)         | 0.312 (n=1)         | 0.880 (n=1)         | 0.923 (n=1)          |
| nisqa        | test_p501     | abl_tau0p1_s0    | 0.428 (n=1)         | 0.848 (n=1)         | 0.882 (n=1)         | 0.350 (n=1)         | 0.871 (n=1)         | 0.915 (n=1)          |
| nisqa        | test_p501     | abl_tau0p5_s0    | 0.360 (n=1)         | 0.857 (n=1)         | 0.869 (n=1)         | 0.285 (n=1)         | 0.883 (n=1)         | 0.906 (n=1)          |
| nisqa        | test_p501     | baseline_noise   | 0.364 ± 0.014 (n=3) | 0.861 ± 0.009 (n=3) | 0.884 ± 0.007 (n=3) | 0.291 ± 0.010 (n=3) | 0.886 ± 0.007 (n=3) | 0.920 ± 0.004 (n=3)  |
| nisqa        | test_p501     | baseline_pgd     | 0.486 ± 0.027 (n=3) | 0.824 ± 0.005 (n=3) | 0.876 ± 0.003 (n=3) | 0.408 ± 0.021 (n=3) | 0.845 ± 0.005 (n=3) | 0.916 ± 0.002 (n=3)  |
| nisqa        | test_p501     | baseline_random  | 0.415 ± 0.020 (n=3) | 0.845 ± 0.013 (n=3) | 0.881 ± 0.006 (n=3) | 0.336 ± 0.022 (n=3) | 0.868 ± 0.012 (n=3) | 0.920 ± 0.003 (n=3)  |
| somos        | test          | R0               | 0.802 ± 0.192 (n=3) | 0.478 ± 0.033 (n=3) | 0.480 ± 0.031 (n=3) | 0.630 ± 0.196 (n=3) | 0.682 ± 0.049 (n=3) | 0.712 ± 0.043 (n=3)  |
| somos        | test          | R1               | 0.870 ± 0.146 (n=3) | 0.471 ± 0.020 (n=3) | 0.470 ± 0.014 (n=3) | 0.682 ± 0.158 (n=3) | 0.677 ± 0.037 (n=3) | 0.702 ± 0.029 (n=3)  |
| somos        | test          | R2               | 0.813 ± 0.098 (n=3) | 0.473 ± 0.005 (n=3) | 0.475 ± 0.013 (n=3) | 0.620 ± 0.102 (n=3) | 0.685 ± 0.006 (n=3) | 0.712 ± 0.004 (n=3)  |
| somos        | test          | R3               | 0.862 ± 0.159 (n=3) | 0.458 ± 0.021 (n=3) | 0.460 ± 0.025 (n=3) | 0.658 ± 0.168 (n=3) | 0.667 ± 0.040 (n=3) | 0.698 ± 0.029 (n=3)  |
| somos        | test          | R3ms             | 0.973 ± 0.155 (n=3) | 0.422 ± 0.034 (n=3) | 0.430 ± 0.025 (n=3) | 0.753 ± 0.130 (n=3) | 0.655 ± 0.010 (n=3) | 0.684 ± 0.009 (n=3)  |
| somos        | test          | abl_headonly_s0  | 0.804 (n=1)         | 0.447 (n=1)         | 0.448 (n=1)         | 0.601 (n=1)         | 0.628 (n=1)         | 0.667 (n=1)          |
| somos        | test          | abl_margin0p1_s0 | 0.850 (n=1)         | 0.460 (n=1)         | 0.449 (n=1)         | 0.637 (n=1)         | 0.674 (n=1)         | 0.686 (n=1)          |
| somos        | test          | abl_margin1p0_s0 | 0.816 (n=1)         | 0.402 (n=1)         | 0.442 (n=1)         | 0.561 (n=1)         | 0.599 (n=1)         | 0.671 (n=1)          |
| somos        | test          | abl_prefinput_s0 | 0.833 (n=1)         | 0.451 (n=1)         | 0.446 (n=1)         | 0.605 (n=1)         | 0.656 (n=1)         | 0.683 (n=1)          |
| somos        | test          | abl_tau0p1_s0    | 0.931 (n=1)         | 0.460 (n=1)         | 0.452 (n=1)         | 0.732 (n=1)         | 0.673 (n=1)         | 0.693 (n=1)          |
| somos        | test          | abl_tau0p5_s0    | 0.832 (n=1)         | 0.447 (n=1)         | 0.441 (n=1)         | 0.636 (n=1)         | 0.657 (n=1)         | 0.682 (n=1)          |
| somos        | test          | baseline_noise   | 0.943 ± 0.117 (n=3) | 0.469 ± 0.009 (n=3) | 0.460 ± 0.012 (n=3) | 0.755 ± 0.119 (n=3) | 0.696 ± 0.009 (n=3) | 0.719 ± 0.002 (n=3)  |
| somos        | test          | baseline_pgd     | 1.010 ± 0.078 (n=3) | 0.446 ± 0.019 (n=3) | 0.450 ± 0.017 (n=3) | 0.807 ± 0.082 (n=3) | 0.692 ± 0.015 (n=3) | 0.712 ± 0.016 (n=3)  |
| somos        | test          | baseline_random  | 0.899 ± 0.175 (n=3) | 0.456 ± 0.019 (n=3) | 0.452 ± 0.021 (n=3) | 0.710 ± 0.170 (n=3) | 0.672 ± 0.019 (n=3) | 0.699 ± 0.011 (n=3)  |
| tmhintqi     | test          | R0               | 3.346 ± 0.473 (n=3) | 0.613 ± 0.013 (n=3) | 0.484 ± 0.049 (n=3) | 2.992 ± 0.499 (n=3) | 0.572 ± 0.044 (n=3) | 0.505 ± 0.112 (n=3)  |
| tmhintqi     | test          | R1               | 4.471 ± 0.457 (n=3) | 0.566 ± 0.010 (n=3) | 0.361 ± 0.029 (n=3) | 4.131 ± 0.483 (n=3) | 0.479 ± 0.034 (n=3) | 0.335 ± 0.060 (n=3)  |
| tmhintqi     | test          | R2               | 4.612 ± 0.166 (n=3) | 0.547 ± 0.038 (n=3) | 0.330 ± 0.032 (n=3) | 4.265 ± 0.137 (n=3) | 0.431 ± 0.090 (n=3) | 0.284 ± 0.096 (n=3)  |
| tmhintqi     | test          | R3               | 4.889 ± 0.910 (n=3) | 0.507 ± 0.044 (n=3) | 0.294 ± 0.050 (n=3) | 4.520 ± 0.943 (n=3) | 0.349 ± 0.084 (n=3) | 0.209 ± 0.087 (n=3)  |
| tmhintqi     | test          | R3ms             | 7.507 ± 1.039 (n=3) | 0.421 ± 0.067 (n=3) | 0.196 ± 0.080 (n=3) | 6.858 ± 1.044 (n=3) | 0.144 ± 0.106 (n=3) | -0.051 ± 0.098 (n=3) |
| tmhintqi     | test          | abl_headonly_s0  | 4.361 (n=1)         | 0.454 (n=1)         | 0.311 (n=1)         | 3.857 (n=1)         | 0.153 (n=1)         | 0.039 (n=1)          |
| tmhintqi     | test          | abl_margin0p1_s0 | 4.061 (n=1)         | 0.592 (n=1)         | 0.437 (n=1)         | 3.677 (n=1)         | 0.549 (n=1)         | 0.489 (n=1)          |
| tmhintqi     | test          | abl_margin1p0_s0 | 7.324 (n=1)         | 0.386 (n=1)         | 0.194 (n=1)         | 6.886 (n=1)         | 0.215 (n=1)         | 0.057 (n=1)          |
| tmhintqi     | test          | abl_prefinput_s0 | 4.998 (n=1)         | 0.532 (n=1)         | 0.304 (n=1)         | 4.653 (n=1)         | 0.416 (n=1)         | 0.239 (n=1)          |
| tmhintqi     | test          | abl_tau0p1_s0    | 4.819 (n=1)         | 0.544 (n=1)         | 0.358 (n=1)         | 4.448 (n=1)         | 0.507 (n=1)         | 0.393 (n=1)          |
| tmhintqi     | test          | abl_tau0p5_s0    | 3.333 (n=1)         | 0.556 (n=1)         | 0.343 (n=1)         | 2.913 (n=1)         | 0.451 (n=1)         | 0.350 (n=1)          |
| tmhintqi     | test          | baseline_noise   | 4.510 ± 0.239 (n=3) | 0.597 ± 0.024 (n=3) | 0.438 ± 0.034 (n=3) | 4.238 ± 0.293 (n=3) | 0.493 ± 0.081 (n=3) | 0.363 ± 0.079 (n=3)  |
| tmhintqi     | test          | baseline_pgd     | 7.985 ± 0.293 (n=3) | 0.521 ± 0.018 (n=3) | 0.376 ± 0.014 (n=3) | 7.564 ± 0.296 (n=3) | 0.356 ± 0.048 (n=3) | 0.263 ± 0.042 (n=3)  |
| tmhintqi     | test          | baseline_random  | 6.496 ± 1.109 (n=3) | 0.563 ± 0.004 (n=3) | 0.347 ± 0.037 (n=3) | 6.322 ± 1.199 (n=3) | 0.511 ± 0.031 (n=3) | 0.397 ± 0.048 (n=3)  |
| urgent24_sqa | test          | R0               | 1.093 ± 0.214 (n=3) | 0.740 ± 0.012 (n=3) | 0.761 ± 0.008 (n=3) | 0.704 ± 0.176 (n=3) | 0.947 ± 0.015 (n=3) | 0.907 ± 0.029 (n=3)  |
| urgent24_sqa | test          | R1               | 1.430 ± 0.124 (n=3) | 0.742 ± 0.011 (n=3) | 0.761 ± 0.010 (n=3) | 0.922 ± 0.113 (n=3) | 0.948 ± 0.001 (n=3) | 0.919 ± 0.005 (n=3)  |
| urgent24_sqa | test          | R2               | 1.412 ± 0.044 (n=3) | 0.739 ± 0.014 (n=3) | 0.759 ± 0.014 (n=3) | 0.892 ± 0.020 (n=3) | 0.947 ± 0.006 (n=3) | 0.914 ± 0.015 (n=3)  |
| urgent24_sqa | test          | R3               | 1.690 ± 0.279 (n=3) | 0.718 ± 0.018 (n=3) | 0.741 ± 0.020 (n=3) | 1.094 ± 0.193 (n=3) | 0.929 ± 0.011 (n=3) | 0.882 ± 0.024 (n=3)  |
| urgent24_sqa | test          | R3ms             | 2.315 ± 0.420 (n=3) | 0.667 ± 0.036 (n=3) | 0.675 ± 0.046 (n=3) | 1.657 ± 0.332 (n=3) | 0.920 ± 0.010 (n=3) | 0.901 ± 0.027 (n=3)  |
| urgent24_sqa | test          | abl_headonly_s0  | 1.343 (n=1)         | 0.704 (n=1)         | 0.728 (n=1)         | 0.875 (n=1)         | 0.861 (n=1)         | 0.829 (n=1)          |
| urgent24_sqa | test          | abl_margin0p1_s0 | 1.396 (n=1)         | 0.721 (n=1)         | 0.746 (n=1)         | 0.882 (n=1)         | 0.938 (n=1)         | 0.906 (n=1)          |
| urgent24_sqa | test          | abl_margin1p0_s0 | 1.991 (n=1)         | 0.710 (n=1)         | 0.732 (n=1)         | 1.264 (n=1)         | 0.900 (n=1)         | 0.838 (n=1)          |
| urgent24_sqa | test          | abl_prefinput_s0 | 1.856 (n=1)         | 0.709 (n=1)         | 0.729 (n=1)         | 1.160 (n=1)         | 0.922 (n=1)         | 0.874 (n=1)          |
| urgent24_sqa | test          | abl_tau0p1_s0    | 1.380 (n=1)         | 0.739 (n=1)         | 0.756 (n=1)         | 0.808 (n=1)         | 0.950 (n=1)         | 0.915 (n=1)          |
| urgent24_sqa | test          | abl_tau0p5_s0    | 1.345 (n=1)         | 0.711 (n=1)         | 0.737 (n=1)         | 0.829 (n=1)         | 0.928 (n=1)         | 0.908 (n=1)          |
| urgent24_sqa | test          | baseline_noise   | 1.112 ± 0.091 (n=3) | 0.764 ± 0.010 (n=3) | 0.777 ± 0.007 (n=3) | 0.697 ± 0.044 (n=3) | 0.972 ± 0.001 (n=3) | 0.956 ± 0.007 (n=3)  |
| urgent24_sqa | test          | baseline_pgd     | 1.817 ± 0.078 (n=3) | 0.744 ± 0.002 (n=3) | 0.753 ± 0.000 (n=3) | 1.281 ± 0.099 (n=3) | 0.948 ± 0.002 (n=3) | 0.963 ± 0.003 (n=3)  |
| urgent24_sqa | test          | baseline_random  | 1.476 ± 0.204 (n=3) | 0.768 ± 0.009 (n=3) | 0.776 ± 0.006 (n=3) | 0.912 ± 0.213 (n=3) | 0.970 ± 0.003 (n=3) | 0.943 ± 0.002 (n=3)  |
| urgent25_sqa | test          | R0               | 1.516 ± 0.243 (n=3) | 0.680 ± 0.015 (n=3) | 0.696 ± 0.012 (n=3) | 1.147 ± 0.223 (n=3) | 0.908 ± 0.007 (n=3) | 0.860 ± 0.012 (n=3)  |
| urgent25_sqa | test          | R1               | 1.858 ± 0.094 (n=3) | 0.697 ± 0.006 (n=3) | 0.706 ± 0.006 (n=3) | 1.469 ± 0.091 (n=3) | 0.899 ± 0.014 (n=3) | 0.894 ± 0.006 (n=3)  |
| urgent25_sqa | test          | R2               | 1.980 ± 0.046 (n=3) | 0.678 ± 0.019 (n=3) | 0.688 ± 0.029 (n=3) | 1.566 ± 0.024 (n=3) | 0.861 ± 0.033 (n=3) | 0.849 ± 0.036 (n=3)  |
| urgent25_sqa | test          | R3               | 2.229 ± 0.326 (n=3) | 0.640 ± 0.027 (n=3) | 0.648 ± 0.042 (n=3) | 1.769 ± 0.273 (n=3) | 0.817 ± 0.050 (n=3) | 0.796 ± 0.080 (n=3)  |
| urgent25_sqa | test          | R3ms             | 2.895 ± 0.347 (n=3) | 0.609 ± 0.029 (n=3) | 0.617 ± 0.040 (n=3) | 2.340 ± 0.316 (n=3) | 0.931 ± 0.009 (n=3) | 0.881 ± 0.023 (n=3)  |
| urgent25_sqa | test          | abl_headonly_s0  | 1.897 (n=1)         | 0.627 (n=1)         | 0.635 (n=1)         | 1.437 (n=1)         | 0.918 (n=1)         | 0.907 (n=1)          |
| urgent25_sqa | test          | abl_margin0p1_s0 | 1.855 (n=1)         | 0.666 (n=1)         | 0.682 (n=1)         | 1.449 (n=1)         | 0.854 (n=1)         | 0.858 (n=1)          |
| urgent25_sqa | test          | abl_margin1p0_s0 | 2.893 (n=1)         | 0.596 (n=1)         | 0.597 (n=1)         | 2.257 (n=1)         | 0.714 (n=1)         | 0.676 (n=1)          |
| urgent25_sqa | test          | abl_prefinput_s0 | 2.398 (n=1)         | 0.643 (n=1)         | 0.640 (n=1)         | 1.877 (n=1)         | 0.765 (n=1)         | 0.803 (n=1)          |
| urgent25_sqa | test          | abl_tau0p1_s0    | 1.897 (n=1)         | 0.684 (n=1)         | 0.689 (n=1)         | 1.477 (n=1)         | 0.879 (n=1)         | 0.879 (n=1)          |
| urgent25_sqa | test          | abl_tau0p5_s0    | 1.718 (n=1)         | 0.653 (n=1)         | 0.666 (n=1)         | 1.315 (n=1)         | 0.842 (n=1)         | 0.863 (n=1)          |
| urgent25_sqa | test          | baseline_noise   | 1.534 ± 0.133 (n=3) | 0.722 ± 0.001 (n=3) | 0.736 ± 0.002 (n=3) | 1.190 ± 0.128 (n=3) | 0.952 ± 0.001 (n=3) | 0.948 ± 0.004 (n=3)  |
| urgent25_sqa | test          | baseline_pgd     | 2.555 ± 0.145 (n=3) | 0.682 ± 0.018 (n=3) | 0.695 ± 0.010 (n=3) | 2.077 ± 0.169 (n=3) | 0.941 ± 0.015 (n=3) | 0.899 ± 0.028 (n=3)  |
| urgent25_sqa | test          | baseline_random  | 2.194 ± 0.265 (n=3) | 0.733 ± 0.009 (n=3) | 0.735 ± 0.006 (n=3) | 1.784 ± 0.266 (n=3) | 0.916 ± 0.005 (n=3) | 0.917 ± 0.009 (n=3)  |
| vmc23        | track1a       | R0               | 0.614 ± 0.112 (n=3) | 0.457 ± 0.030 (n=3) | 0.410 ± 0.025 (n=3) | 0.374 ± 0.102 (n=3) | 0.763 ± 0.019 (n=3) | 0.710 ± 0.041 (n=3)  |
| vmc23        | track1a       | R1               | 0.571 ± 0.055 (n=3) | 0.451 ± 0.039 (n=3) | 0.415 ± 0.035 (n=3) | 0.320 ± 0.050 (n=3) | 0.749 ± 0.023 (n=3) | 0.699 ± 0.048 (n=3)  |
| vmc23        | track1a       | R2               | 0.592 ± 0.038 (n=3) | 0.431 ± 0.032 (n=3) | 0.397 ± 0.029 (n=3) | 0.334 ± 0.033 (n=3) | 0.737 ± 0.019 (n=3) | 0.694 ± 0.030 (n=3)  |
| vmc23        | track1a       | R3               | 0.637 ± 0.030 (n=3) | 0.423 ± 0.023 (n=3) | 0.393 ± 0.029 (n=3) | 0.380 ± 0.026 (n=3) | 0.705 ± 0.029 (n=3) | 0.677 ± 0.029 (n=3)  |
| vmc23        | track1a       | R3ms             | 0.981 ± 0.462 (n=3) | 0.315 ± 0.084 (n=3) | 0.298 ± 0.047 (n=3) | 0.665 ± 0.417 (n=3) | 0.505 ± 0.139 (n=3) | 0.538 ± 0.013 (n=3)  |
| vmc23        | track1a       | abl_headonly_s0  | 0.609 (n=1)         | 0.460 (n=1)         | 0.415 (n=1)         | 0.355 (n=1)         | 0.692 (n=1)         | 0.627 (n=1)          |
| vmc23        | track1a       | abl_margin0p1_s0 | 0.624 (n=1)         | 0.441 (n=1)         | 0.415 (n=1)         | 0.358 (n=1)         | 0.739 (n=1)         | 0.688 (n=1)          |
| vmc23        | track1a       | abl_margin1p0_s0 | 0.599 (n=1)         | 0.444 (n=1)         | 0.419 (n=1)         | 0.352 (n=1)         | 0.717 (n=1)         | 0.695 (n=1)          |
| vmc23        | track1a       | abl_prefinput_s0 | 0.574 (n=1)         | 0.449 (n=1)         | 0.431 (n=1)         | 0.324 (n=1)         | 0.698 (n=1)         | 0.689 (n=1)          |
| vmc23        | track1a       | abl_tau0p1_s0    | 0.565 (n=1)         | 0.454 (n=1)         | 0.417 (n=1)         | 0.310 (n=1)         | 0.750 (n=1)         | 0.681 (n=1)          |
| vmc23        | track1a       | abl_tau0p5_s0    | 0.560 (n=1)         | 0.448 (n=1)         | 0.421 (n=1)         | 0.294 (n=1)         | 0.734 (n=1)         | 0.704 (n=1)          |
| vmc23        | track1a       | baseline_noise   | 0.627 ± 0.164 (n=3) | 0.455 ± 0.061 (n=3) | 0.408 ± 0.056 (n=3) | 0.379 ± 0.161 (n=3) | 0.793 ± 0.017 (n=3) | 0.726 ± 0.028 (n=3)  |
| vmc23        | track1a       | baseline_pgd     | 0.690 ± 0.105 (n=3) | 0.473 ± 0.029 (n=3) | 0.402 ± 0.025 (n=3) | 0.429 ± 0.099 (n=3) | 0.763 ± 0.019 (n=3) | 0.674 ± 0.038 (n=3)  |
| vmc23        | track1a       | baseline_random  | 0.600 ± 0.043 (n=3) | 0.438 ± 0.018 (n=3) | 0.397 ± 0.026 (n=3) | 0.345 ± 0.045 (n=3) | 0.782 ± 0.024 (n=3) | 0.727 ± 0.031 (n=3)  |
| vmc23        | track1b       | R0               | 2.157 ± 0.405 (n=3) | 0.267 ± 0.023 (n=3) | 0.238 ± 0.016 (n=3) | 1.903 ± 0.381 (n=3) | 0.410 ± 0.033 (n=3) | 0.494 ± 0.071 (n=3)  |
| vmc23        | track1b       | R1               | 2.496 ± 0.058 (n=3) | 0.284 ± 0.038 (n=3) | 0.260 ± 0.033 (n=3) | 2.245 ± 0.066 (n=3) | 0.383 ± 0.038 (n=3) | 0.493 ± 0.037 (n=3)  |
| vmc23        | track1b       | R2               | 2.758 ± 0.229 (n=3) | 0.292 ± 0.065 (n=3) | 0.279 ± 0.063 (n=3) | 2.502 ± 0.207 (n=3) | 0.387 ± 0.122 (n=3) | 0.492 ± 0.199 (n=3)  |
| vmc23        | track1b       | R3               | 3.005 ± 0.249 (n=3) | 0.274 ± 0.071 (n=3) | 0.267 ± 0.067 (n=3) | 2.741 ± 0.220 (n=3) | 0.352 ± 0.137 (n=3) | 0.466 ± 0.164 (n=3)  |
| vmc23        | track1b       | R3ms             | 3.646 ± 1.090 (n=3) | 0.277 ± 0.064 (n=3) | 0.247 ± 0.057 (n=3) | 3.374 ± 1.066 (n=3) | 0.332 ± 0.082 (n=3) | 0.334 ± 0.115 (n=3)  |
| vmc23        | track1b       | abl_headonly_s0  | 2.822 (n=1)         | 0.287 (n=1)         | 0.282 (n=1)         | 2.526 (n=1)         | 0.402 (n=1)         | 0.566 (n=1)          |
| vmc23        | track1b       | abl_margin0p1_s0 | 2.754 (n=1)         | 0.244 (n=1)         | 0.233 (n=1)         | 2.499 (n=1)         | 0.322 (n=1)         | 0.385 (n=1)          |
| vmc23        | track1b       | abl_margin1p0_s0 | 4.050 (n=1)         | 0.289 (n=1)         | 0.281 (n=1)         | 3.664 (n=1)         | 0.341 (n=1)         | 0.424 (n=1)          |
| vmc23        | track1b       | abl_prefinput_s0 | 3.140 (n=1)         | 0.217 (n=1)         | 0.206 (n=1)         | 2.847 (n=1)         | 0.259 (n=1)         | 0.397 (n=1)          |
| vmc23        | track1b       | abl_tau0p1_s0    | 2.474 (n=1)         | 0.246 (n=1)         | 0.223 (n=1)         | 2.221 (n=1)         | 0.319 (n=1)         | 0.387 (n=1)          |
| vmc23        | track1b       | abl_tau0p5_s0    | 2.445 (n=1)         | 0.256 (n=1)         | 0.255 (n=1)         | 2.204 (n=1)         | 0.315 (n=1)         | 0.426 (n=1)          |
| vmc23        | track1b       | baseline_noise   | 1.970 ± 0.188 (n=3) | 0.306 ± 0.024 (n=3) | 0.263 ± 0.023 (n=3) | 1.723 ± 0.204 (n=3) | 0.497 ± 0.059 (n=3) | 0.559 ± 0.083 (n=3)  |
| vmc23        | track1b       | baseline_pgd     | 3.140 ± 0.348 (n=3) | 0.331 ± 0.011 (n=3) | 0.276 ± 0.014 (n=3) | 2.873 ± 0.349 (n=3) | 0.373 ± 0.009 (n=3) | 0.312 ± 0.048 (n=3)  |
| vmc23        | track1b       | baseline_random  | 2.354 ± 0.029 (n=3) | 0.306 ± 0.034 (n=3) | 0.269 ± 0.034 (n=3) | 2.111 ± 0.036 (n=3) | 0.419 ± 0.062 (n=3) | 0.453 ± 0.079 (n=3)  |
| vmc23        | track2        | R0               | 1.568 ± 0.245 (n=3) | 0.494 ± 0.010 (n=3) | 0.478 ± 0.014 (n=3) | 1.246 ± 0.241 (n=3) | 0.773 ± 0.011 (n=3) | 0.734 ± 0.016 (n=3)  |
| vmc23        | track2        | R1               | 1.844 ± 0.353 (n=3) | 0.477 ± 0.036 (n=3) | 0.455 ± 0.032 (n=3) | 1.509 ± 0.346 (n=3) | 0.755 ± 0.011 (n=3) | 0.695 ± 0.009 (n=3)  |
| vmc23        | track2        | R2               | 1.844 ± 0.208 (n=3) | 0.474 ± 0.015 (n=3) | 0.453 ± 0.011 (n=3) | 1.507 ± 0.194 (n=3) | 0.744 ± 0.008 (n=3) | 0.688 ± 0.014 (n=3)  |
| vmc23        | track2        | R3               | 1.992 ± 0.291 (n=3) | 0.457 ± 0.029 (n=3) | 0.441 ± 0.023 (n=3) | 1.663 ± 0.286 (n=3) | 0.735 ± 0.012 (n=3) | 0.662 ± 0.022 (n=3)  |
| vmc23        | track2        | R3ms             | 2.457 ± 0.323 (n=3) | 0.362 ± 0.010 (n=3) | 0.352 ± 0.015 (n=3) | 2.020 ± 0.246 (n=3) | 0.719 ± 0.037 (n=3) | 0.658 ± 0.036 (n=3)  |
| vmc23        | track2        | abl_headonly_s0  | 1.973 (n=1)         | 0.347 (n=1)         | 0.341 (n=1)         | 1.563 (n=1)         | 0.638 (n=1)         | 0.598 (n=1)          |
| vmc23        | track2        | abl_margin0p1_s0 | 1.885 (n=1)         | 0.440 (n=1)         | 0.422 (n=1)         | 1.561 (n=1)         | 0.718 (n=1)         | 0.652 (n=1)          |
| vmc23        | track2        | abl_margin1p0_s0 | 2.078 (n=1)         | 0.454 (n=1)         | 0.431 (n=1)         | 1.724 (n=1)         | 0.737 (n=1)         | 0.659 (n=1)          |
| vmc23        | track2        | abl_prefinput_s0 | 2.042 (n=1)         | 0.462 (n=1)         | 0.448 (n=1)         | 1.707 (n=1)         | 0.745 (n=1)         | 0.685 (n=1)          |
| vmc23        | track2        | abl_tau0p1_s0    | 1.651 (n=1)         | 0.488 (n=1)         | 0.459 (n=1)         | 1.329 (n=1)         | 0.741 (n=1)         | 0.669 (n=1)          |
| vmc23        | track2        | abl_tau0p5_s0    | 1.559 (n=1)         | 0.496 (n=1)         | 0.474 (n=1)         | 1.260 (n=1)         | 0.746 (n=1)         | 0.672 (n=1)          |
| vmc23        | track2        | baseline_noise   | 1.698 ± 0.185 (n=3) | 0.475 ± 0.023 (n=3) | 0.451 ± 0.018 (n=3) | 1.386 ± 0.183 (n=3) | 0.758 ± 0.011 (n=3) | 0.694 ± 0.016 (n=3)  |
| vmc23        | track2        | baseline_pgd     | 2.449 ± 0.273 (n=3) | 0.326 ± 0.009 (n=3) | 0.320 ± 0.020 (n=3) | 2.010 ± 0.276 (n=3) | 0.630 ± 0.002 (n=3) | 0.569 ± 0.017 (n=3)  |
| vmc23        | track2        | baseline_random  | 1.519 ± 0.235 (n=3) | 0.490 ± 0.011 (n=3) | 0.459 ± 0.010 (n=3) | 1.196 ± 0.256 (n=3) | 0.750 ± 0.021 (n=3) | 0.687 ± 0.021 (n=3)  |
| vmc23        | track3        | R0               | 2.543 ± 0.405 (n=3) | 0.610 ± 0.018 (n=3) | 0.484 ± 0.040 (n=3) | 2.195 ± 0.403 (n=3) | 0.756 ± 0.029 (n=3) | 0.604 ± 0.065 (n=3)  |
| vmc23        | track3        | R1               | 3.470 ± 0.396 (n=3) | 0.527 ± 0.042 (n=3) | 0.390 ± 0.040 (n=3) | 3.099 ± 0.399 (n=3) | 0.661 ± 0.044 (n=3) | 0.494 ± 0.036 (n=3)  |
| vmc23        | track3        | R2               | 3.695 ± 0.129 (n=3) | 0.500 ± 0.042 (n=3) | 0.372 ± 0.014 (n=3) | 3.309 ± 0.108 (n=3) | 0.647 ± 0.031 (n=3) | 0.502 ± 0.022 (n=3)  |
| vmc23        | track3        | R3               | 4.015 ± 0.798 (n=3) | 0.415 ± 0.150 (n=3) | 0.295 ± 0.139 (n=3) | 3.605 ± 0.762 (n=3) | 0.529 ± 0.199 (n=3) | 0.385 ± 0.216 (n=3)  |
| vmc23        | track3        | R3ms             | 6.389 ± 1.015 (n=3) | 0.260 ± 0.074 (n=3) | 0.140 ± 0.065 (n=3) | 5.929 ± 1.011 (n=3) | 0.297 ± 0.088 (n=3) | 0.146 ± 0.076 (n=3)  |
| vmc23        | track3        | abl_headonly_s0  | 3.432 (n=1)         | 0.412 (n=1)         | 0.342 (n=1)         | 2.985 (n=1)         | 0.546 (n=1)         | 0.407 (n=1)          |
| vmc23        | track3        | abl_margin0p1_s0 | 3.196 (n=1)         | 0.550 (n=1)         | 0.460 (n=1)         | 2.827 (n=1)         | 0.701 (n=1)         | 0.602 (n=1)          |
| vmc23        | track3        | abl_margin1p0_s0 | 6.380 (n=1)         | 0.049 (n=1)         | -0.028 (n=1)        | 5.820 (n=1)         | 0.016 (n=1)         | -0.075 (n=1)         |
| vmc23        | track3        | abl_prefinput_s0 | 4.258 (n=1)         | 0.348 (n=1)         | 0.205 (n=1)         | 3.840 (n=1)         | 0.434 (n=1)         | 0.292 (n=1)          |
| vmc23        | track3        | abl_tau0p1_s0    | 3.691 (n=1)         | 0.449 (n=1)         | 0.345 (n=1)         | 3.284 (n=1)         | 0.579 (n=1)         | 0.462 (n=1)          |
| vmc23        | track3        | abl_tau0p5_s0    | 2.821 (n=1)         | 0.443 (n=1)         | 0.339 (n=1)         | 2.423 (n=1)         | 0.586 (n=1)         | 0.485 (n=1)          |
| vmc23        | track3        | baseline_noise   | 3.449 ± 0.239 (n=3) | 0.558 ± 0.034 (n=3) | 0.426 ± 0.033 (n=3) | 3.118 ± 0.249 (n=3) | 0.656 ± 0.052 (n=3) | 0.488 ± 0.035 (n=3)  |
| vmc23        | track3        | baseline_pgd     | 6.538 ± 0.326 (n=3) | 0.469 ± 0.025 (n=3) | 0.309 ± 0.025 (n=3) | 6.111 ± 0.316 (n=3) | 0.570 ± 0.032 (n=3) | 0.358 ± 0.022 (n=3)  |
| vmc23        | track3        | baseline_random  | 4.909 ± 0.935 (n=3) | 0.498 ± 0.011 (n=3) | 0.328 ± 0.020 (n=3) | 4.521 ± 0.960 (n=3) | 0.629 ± 0.036 (n=3) | 0.450 ± 0.033 (n=3)  |

In [9]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('r0_bvcc', 'ood_sys_srcc', 'bvcc_sys'))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

19 numbers; max |diff| = 0.0000; identical: 19


,paper,reproduced,abs_diff
bvcc_sys_srcc_R3_minus_R0,-0.002,-0.002,0.0
ood_sys_srcc_R0,0.731,0.731,0.0
ood_sys_srcc_R1,0.690,0.690,0.0
ood_sys_srcc_R2,0.682,0.682,0.0
ood_sys_srcc_R3,0.637,0.637,0.0
ood_sys_srcc_R3ms,0.552,0.552,0.0
ood_sys_srcc_abl_headonly_s0,0.629,0.629,0.0
ood_sys_srcc_abl_margin0p1_s0,0.686,0.686,0.0
ood_sys_srcc_abl_margin1p0_s0,0.565,0.565,0.0
ood_sys_srcc_abl_prefinput_s0,0.639,0.639,0.0
